# Phenotype ↔ Gene Relation-Wise Merge

Merges Phenotype–Gene triples from PrimeKG; deduplicates by `(head, relation, tail)`;
drops rows with unresolved `head_detail_name`; and saves the result.

## 0. Configuration

In [1]:
import os
import pandas as pd

BASE_DIR  = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR  = BASE_DIR + 'processed_data/'
DB_DIR     = BASE_DIR + 'data_collection/databases_for_mapping/'

OUT_PATH  = BASE_DIR + 'processed_data_relation_wise_merge/generalised/PHENOTYPE_GENE/ALL_PHENOTYPE_GENE.csv'

REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1. Load KG Sources

### 1a. PrimeKG

In [15]:
PrimeKG_Phenotype_Gene = pd.read_csv(PROC_DIR + 'PrimeKG/PrimeKG_Phenotype_Gene.csv')
PrimeKG_Phenotype_Gene.columns = PrimeKG_Phenotype_Gene.columns.str.lower()
print(f"PrimeKG: {len(PrimeKG_Phenotype_Gene):,} rows | columns: {list(PrimeKG_Phenotype_Gene.columns)}")
PrimeKG_Phenotype_Gene['kg_type'] = 'Generalised'
PrimeKG_Phenotype_Gene['species'] = 'Homo species'

PrimeKG_Phenotype_Gene.head(2)

PrimeKG: 3,330 rows | columns: ['head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type', 'kg_source', 'head_id_is', 'tail_id_is', 'head_detail_name', 'tail_alias_mapped', 'tail_detail_name', 'tail_ens']


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_alias_mapped,tail_detail_name,tail_ens,kg_type,species
0,HP:0002240,Phenotype_Gene,A1BG,Phenotype,associated with,Gene,PrimeKG,HPO_ID,NCBI_ID,Hepatomegaly,A1BG,alpha-1-B glycoprotein,ENSG00000121410,Generalised,Homo species
1,HP:0002240,Phenotype_Gene,AHR,Phenotype,associated with,Gene,PrimeKG,HPO_ID,NCBI_ID,Hepatomegaly,AHR,aryl hydrocarbon receptor,ENSG00000106546,Generalised,Homo species


### 1b. Pheknowlator

In [16]:
Pheknowlator = pd.read_csv(PROC_DIR + 'pheknowlator/Phenotype_Gene.csv')
Pheknowlator.columns = Pheknowlator.columns.str.lower()
print(f"Pheknowlator: {len(Pheknowlator):,} rows | columns: {list(Pheknowlator.columns)}")
Pheknowlator['kg_type'] = 'Generalised'
Pheknowlator['species'] = 'Homo species'
Pheknowlator.head(2)

Pheknowlator: 150,653 rows | columns: ['head', 'relation', 'tail', 'head_type', 'tail_type', 'head_detail_name', 'tail_detail_name', 'head_id_is', 'tail_id_is', 'kg_source']


,head,relation,tail,head_type,tail_type,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_source,kg_type,species
0,HP:0000028,Phenotype_Gene,B3GALNT2,Phenotype,Gene,Cryptorchidism,"beta-1,3-N-acetylgalactosaminyltransferase 2",HPO_ID,NCBI_ID,pheknowlator,Generalised,Homo species
1,HP:0002920,Phenotype_Gene,CDH23,Phenotype,Gene,Decreased circulating ACTH concentration,cadherin related 23,HPO_ID,NCBI_ID,pheknowlator,Generalised,Homo species


## 2. Check and Fix Duplicate Columns

In [17]:
SOURCE_DFS = [('PrimeKG_Phenotype_Gene', PrimeKG_Phenotype_Gene),
             ('Pheknowlator', Pheknowlator)]

for name, df in SOURCE_DFS:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    print(f"[{name}] {'duplicate columns: ' + str(dupes) if dupes else '✓ no duplicates'}")

[PrimeKG_Phenotype_Gene] ✓ no duplicates
[Pheknowlator] ✓ no duplicates


## 3. Align Schemas and Concatenate

In [18]:
CONCAT_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

df_list = []
for name, df in SOURCE_DFS:
    tmp = df.loc[:, ~df.columns.duplicated()].copy()
    for col in CONCAT_COLS:
        if col not in tmp.columns:
            tmp[col] = pd.NA
    df_list.append(tmp[CONCAT_COLS])

final_df = pd.concat(df_list, ignore_index=True)
print(f"Combined: {len(final_df):,} rows")
final_df.head(2)

Combined: 153,983 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,HP:0002240,Phenotype_Gene,A1BG,Phenotype,associated with,Gene,PrimeKG,HPO_ID,NCBI_ID,Hepatomegaly,alpha-1-B glycoprotein,Generalised,Homo species
1,HP:0002240,Phenotype_Gene,AHR,Phenotype,associated with,Gene,PrimeKG,HPO_ID,NCBI_ID,Hepatomegaly,aryl hydrocarbon receptor,Generalised,Homo species


## 4. Standardise Fixed-Value Columns

In [19]:
final_df['head_type']  = 'Phenotype'
final_df['tail_type']  = 'Gene'
final_df['relation']   = 'Phenotype_Gene'
final_df['head_id_is'] = 'HPO'
final_df['tail_id_is'] = 'NCBI_ID'

print("Unique relation:",  set(final_df['relation']))
print("Unique kg_source:", set(final_df['kg_source']))

Unique relation: {'Phenotype_Gene'}
Unique kg_source: {'PrimeKG', 'pheknowlator'}


## 5. Deduplicate and Add Schema Columns

In [20]:
GROUP_COLS = ['head', 'relation', 'tail']

def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

final_df_dedup = final_df.groupby(GROUP_COLS, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':         merge_sources,
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'kg_type': merge_sources,
    'species': 'first',
})
print(f"After deduplication: {len(final_df_dedup):,} rows")

# Drop rows with unresolved head_detail_name
before = len(final_df_dedup)
final_df_dedup = final_df_dedup[~final_df_dedup['head_detail_name'].isna()].reset_index(drop=True)
print(f"Dropped {before - len(final_df_dedup):,} rows with missing head_detail_name")

final_df_dedup = final_df_dedup[REQUIRED_COLS]
final_df_dedup.head()

After deduplication: 153,582 rows
Dropped 0 rows with missing head_detail_name


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,HP:0000002,Phenotype_Gene,ANOS1,Phenotype,None,Gene,pheknowlator,HPO,NCBI_ID,Abnormality of body height,anosmin 1,Generalised,Homo species
1,HP:0000002,Phenotype_Gene,CHD7,Phenotype,None,Gene,pheknowlator,HPO,NCBI_ID,Abnormality of body height,chromodomain helicase DNA binding protein 7,Generalised,Homo species
2,HP:0000002,Phenotype_Gene,DUSP6,Phenotype,None,Gene,pheknowlator,HPO,NCBI_ID,Abnormality of body height,dual specificity phosphatase 6,Generalised,Homo species
3,HP:0000002,Phenotype_Gene,FGF17,Phenotype,None,Gene,pheknowlator,HPO,NCBI_ID,Abnormality of body height,fibroblast growth factor 17,Generalised,Homo species
4,HP:0000002,Phenotype_Gene,FGF8,Phenotype,None,Gene,pheknowlator,HPO,NCBI_ID,Abnormality of body height,fibroblast growth factor 8,Generalised,Homo species


## 6. QC — NaN Check

In [21]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,150256,0,150256
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


```
Note: NaN in relation_type because phenknowlator has not provided it's relationtype label
```

## 7. Save Output

In [22]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)

final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 153,582 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/PHENOTYPE_GENE/ALL_PHENOTYPE_GENE.csv
